In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import poisson
from sklearn.linear_model import PoissonRegressor, Ridge
from scipy.stats import chi2
from sklearn.metrics import mean_poisson_deviance

# Read the file through the correct path
df = pd.read_csv(r'C:\Users\matia\data\2023_Yellow_Taxi_Trip_Data_20260417.csv')

# Our next task is to properly handle the datetime data
df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'], format='%m/%d/%Y %I:%M:%S %p')

# Make sure we aren't dealing with any empty rows
df = df.dropna(subset=['tpep_pickup_datetime'])

# We then need to group our timestamps into hourly pickups
df['hour'] = df['tpep_pickup_datetime'].dt.floor('H') # this rounds the time to the lowest nearby hour
df['hour_of_day'] = df['tpep_pickup_datetime'].dt.hour

# We then need to get an amount of pickups for each hour
hourly_counts = df.groupby('hour').size().reset_index(name='count')
hourly_counts['hour_of_day'] = hourly_counts['hour'].dt.hour

# Calculate mean and variance for the whole data set
whole_mean = hourly_counts['count'].mean()
whole_variance = hourly_counts['count'].var()

# Show that we cant model the entire data set by one poisson distribution (variance>>mean)
print("Mean for the whole dataset:", whole_mean)
print("Variance for the whole dataset:", whole_variance)

# We take a different lambda for each hour of the day so we can use the Poisson Regression
lambda_by_hour = hourly_counts.groupby('hour_of_day')['count'].mean()

print("Lambda per hour:")
print(lambda_by_hour)

# Create a plot of the lamdas per hour
plt.plot(lambda_by_hour.index, lambda_by_hour.values, marker='o')
plt.title("Estimated Poisson λ by Hour of Day")
plt.xlabel("Hour of Day")
plt.ylabel("Mean Pickups (λ)")
plt.show()

X = pd.get_dummies(hourly_counts['hour_of_day'], drop_first=True)
y = hourly_counts['count']

model = PoissonRegressor()
model.fit(X, y)

# Apply the same encoding to the 24 hours for prediction
hours_df = pd.DataFrame({'hour_of_day': np.arange(24)})
X_pred = pd.get_dummies(hours_df['hour_of_day'], drop_first=True)
X_pred = X_pred.reindex(columns=X.columns, fill_value=0)  # make sure columns match exactly

fitted_lambda = model.predict(X_pred)

# Create the empircal plot with the overlay
plt.figure()
plt.plot(np.arange(24), fitted_lambda, 'r-', label='Poisson Regression (fitted λ)')
plt.plot(lambda_by_hour.index, lambda_by_hour.values, 'bo', label='Observed mean counts')
plt.title("Poisson Regression Fit by Hour of Day")
plt.xlabel("Hour of Day")
plt.ylabel("Expected Pickups (λ)")
plt.legend()
plt.show()


for h in range(24):
    lam = fitted_lambda[h]
    observed = hourly_counts.loc[hourly_counts['hour_of_day'] == h, 'count']
    
    # Sort the observed values
    observed_sorted = np.sort(observed)
    
    # Empirical CDF — probability of each observed value
    empirical_cdf = np.arange(1, len(observed_sorted) + 1) / len(observed_sorted)
    
    # Theoretical CDF — Poisson CDF at each observed value
    theoretical_cdf = poisson.cdf(observed_sorted, mu=lam)
    
    plt.figure()
    plt.scatter(theoretical_cdf, empirical_cdf, color='steelblue', s=10)
    plt.plot([0, 1], [0, 1], 'r--', label='Perfect fit')
    plt.title(f'P-P Plot — Hour {h}:00 (λ={lam:.0f})')
    plt.xlabel('Theoretical CDF (Poisson)')
    plt.ylabel('Empirical CDF')
    plt.legend()
    plt.show()

# Calculating the AIC and BIC for the plot in the document
y_pred = model.predict(X)
log_likelihood = np.sum(y * np.log(y_pred) - y_pred)
k = X.shape[1] + 1
n = len(y)

aic_poisson = 2 * k - 2 * log_likelihood
bic_poisson = k * np.log(n) - 2 * log_likelihood

print(f"Poisson AIC: {aic_poisson:.2f}")
print(f"Poisson BIC: {bic_poisson:.2f}")

<class 'ModuleNotFoundError'>: No module named 'pandas'